# 06 · Shot Budget: what kernel concentration actually costs

Notebook 05 showed that kernel concentration does **not** predict the achievable AUC: with exact kernels (and even with 1000 shots) concentrated kernels still classify well.

**Hypothesis:** concentration sets the **number of shots** needed to reach that AUC.
A kernel entry estimated with N shots has noise $\sqrt{K(1-K)/N}$, while the jet-to-jet signal is $\sqrt{\mathrm{Var}(K)}$.
Noise exceeds signal below

$$N^* \approx \frac{\langle K(1-K)\rangle}{\mathrm{Var}(K)}$$

(mean and variance over off-diagonal training entries). The SVM averages over hundreds of support vectors, so the actual breakdown may occur below $N^*$.

**Test:** AUC vs shots per kernel entry, from 3 to 10 000, for configurations with very different concentration.

In [ ]:
import sys
sys.path.insert(0, "src")

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC

from jet_data import load_jets, select_quark_gluon, feature_matrix, event_level_split, balanced_subsample, BOOKKEEPING
from kernels import KERNELS, preprocess, shot_noise
from figure_style import set_style, save, COLORS

set_style()
SEED = 42
SEEDS = [0, 1, 2, 3, 4]
N_PER_CLASS = 200
SHOT_GRID = [3, 10, 30, 100, 300, 1000, 3000, 10000]
FEATURES = ("QG_mult", "QG_ptD", "QG_axis2")
RESULTS = Path("results")
baselines = json.load(open(RESULTS / "baselines.json"))

# (encoding, preprocessing, line style) — from most to least concentrated, plus the sign-blind case
CONFIGS = [
    ("amplitude", "raw", "-"),
    ("angle", "minmax", "-"),
    ("amplitude", "minmax", "--"),
    ("angle", "robust_yj", "--"),
    ("amplitude", "standard", ":"),
]
ENC_COLOR = {"amplitude": COLORS["series1"], "angle": COLORS["series2"]}

raw = load_jets(features=FEATURES, extra=BOOKKEEPING)
jets = select_quark_gluon(raw, require_loose_id=True)
y = jets["label"]
X = feature_matrix(jets, features=FEATURES)
train, test = event_level_split(jets, test_fraction=0.3, seed=SEED)

## 1. Run (about 1–2 minutes)

In [ ]:
results = {}
for enc, prep, _ in CONFIGS:
    kernel = KERNELS[enc]
    auc = {n: [] for n in SHOT_GRID}
    auc["exact"] = []
    n_star = []
    for s in SEEDS:
        idx = balanced_subsample(y, train, N_PER_CLASS, seed=s)
        A, T = preprocess(prep, X[idx], X[test])
        K_tr, K_te = kernel(A, A), kernel(T, A)
        off = K_tr[~np.eye(len(idx), dtype=bool)]
        n_star.append(np.mean(off * (1 - off)) / off.var())

        svc = SVC(kernel="precomputed", C=1.0).fit(K_tr, y[idx])
        auc["exact"].append(roc_auc_score(y[test], svc.decision_function(K_te)))
        rng = np.random.default_rng(2000 + s)
        for n in SHOT_GRID:
            svc = SVC(kernel="precomputed", C=1.0).fit(shot_noise(K_tr, n, rng, symmetric=True), y[idx])
            auc[n].append(roc_auc_score(y[test], svc.decision_function(shot_noise(K_te, n, rng))))
    results[(enc, prep)] = {"auc": auc, "n_star": float(np.mean(n_star))}
    print(f"{enc:9s} {prep:9s}  N* ≈ {np.mean(n_star):8.1f}   exact AUC {np.mean(auc['exact']):.3f}")

json.dump({f"{e}/{p}": {"n_star": r["n_star"], "auc": {str(k): v for k, v in r["auc"].items()}}
           for (e, p), r in results.items()},
          open(RESULTS / "shot_budget.json", "w"), indent=2)
print("saved results/shot_budget.json")

## 2. AUC vs shots

Vertical ticks mark the predicted $N^*$ for each configuration (same colour and line style).

In [ ]:
PREP_LABEL = {"raw": "raw", "standard": "standardised", "robust_yj": "robust + Yeo–Johnson", "minmax": "min-max"}
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for enc, prep, ls in CONFIGS:
    r = results[(enc, prep)]
    mean = np.array([np.mean(r["auc"][n]) for n in SHOT_GRID])
    std = np.array([np.std(r["auc"][n]) for n in SHOT_GRID])
    color = ENC_COLOR[enc]
    ax.plot(SHOT_GRID, mean, ls, color=color, marker="o", markersize=5, label=f"{enc}, {PREP_LABEL[prep]}")
    ax.fill_between(SHOT_GRID, mean - std, mean + std, color=color, alpha=0.12, linewidth=0)
    ax.axvline(r["n_star"], ymin=0, ymax=0.06, color=color, linestyle=ls, linewidth=2)
rbf = np.mean(baselines["rbf_svm_sub_auc"])
ax.axhline(rbf, color=COLORS["reference"], linestyle="--", linewidth=1)
ax.text(SHOT_GRID[-1] * 1.15, rbf, "classical\nRBF-SVM", va="center", fontsize=8.5, color=COLORS["reference"])
ax.axhline(0.5, color=COLORS["reference"], linewidth=0.8, alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel("shots per kernel entry")
ax.set_ylabel("test AUC (quark vs gluon)")
ax.set_title("Concentrated kernels need more shots", loc="left")
ax.legend(fontsize=8.5, loc="lower right")
fig.tight_layout()
save(fig, "fig08_shot_budget")
plt.show()

## 3. Where does each configuration break?

Breakdown shots = smallest N at which the mean AUC is within 0.01 of its exact-kernel value.

In [ ]:
print(f"{'configuration':24s} {'N* (predicted)':>15s} {'shots to reach exact - 0.01':>28s}")
for enc, prep, _ in CONFIGS:
    r = results[(enc, prep)]
    target = np.mean(r["auc"]["exact"]) - 0.01
    reached = [n for n in SHOT_GRID if np.mean(r["auc"][n]) >= target]
    print(f"{enc + '/' + prep:24s} {r['n_star']:15.1f} {str(reached[0]) if reached else '> ' + str(SHOT_GRID[-1]):>28s}")

## 4. Summary for the research log

In [ ]:
for enc, prep, _ in CONFIGS:
    r = results[(enc, prep)]
    print(f"- {enc}/{prep}: N* {r['n_star']:.1f}; AUC exact {np.mean(r['auc']['exact']):.3f}, "
          + ", ".join(f"{n}: {np.mean(r['auc'][n]):.3f}" for n in (10, 100, 1000)))